[![pythonista.io](img/pythonista.png)](https://www.pythonista.io)

# Build de contenedores y *runners* endurecidos en *GitHub Actions*.Esta sesión define el proceso operativo de *build* del curso usando *GitHub Actions* como plano de control.Se comparan *runners* hosted y self-hosted con enfoque de seguridad, trazabilidad y costo operativo.

## Objetivos.- Diseñar un *workflow* de *build* reproducible y trazable.- Diferenciar cuándo usar *runners* hosted y cuándo self-hosted.- Aplicar controles de hardening mínimos para ambos tipos de *runner*.- Preparar artefactos para promoción hacia test (*Linux*) y prod (*GCP*).

## Narrativa operativa de este bloque.1. Se construye la imagen de forma consistente desde *GitHub*.2. Se publica en un registro con etiqueta inmutable por *commit*.3. Se promueve la misma imagen a distintos entornos.4. El entorno de test puede desplegar en *Linux*; prod puede desplegar en *GCP*.La idea clave: cambiar entorno no debe implicar reconstruir la imagen.

## Hosted vs self-hosted *runners*.### Hosted- Menor carga operativa.- Buen punto de partida para la mayoría de *pipelines*.- Aislamiento efímero administrado por *GitHub*.### Self-hosted- Útiles cuando se requiere acceso a red privada o dependencias internas.- Exigen parches, monitoreo, hardening y rotación de credenciales.- Si no son efímeros, aumenta el riesgo de persistencia entre *jobs*.Regla práctica del curso: hosted por defecto; self-hosted solo por necesidad técnica justificada.

## Controles de hardening recomendados.- Permisos mínimos en GITHUB_TOKEN (permissions por *job*).- Versionado/pinning de actions (idealmente por SHA en entornos críticos).- Build con etiquetas inmutables (por ejemplo, github.sha).- Escaneo de Dockerfile e imagen antes de push.- Evitar secretos estáticos cuando sea posible (*OIDC* para nube).- En self-hosted: ejecución efímera, limpieza post-*job* y segmentación de red.- Protección de ramas y entornos con aprobaciones.

## *Hadolint*: linter para *Dockerfiles*.

[*Hadolint*](https://github.com/hadolint/hadolint) (*Haskell Dockerfile Linter*) analiza un *Dockerfile* antes de construir la imagen y señala instrucciones que violan buenas prácticas de construcción o introducen riesgos de seguridad. Combina dos motores:

- **DL** (*Dockerfile Lint*): reglas específicas de sintaxis y estructura del *Dockerfile*.
- **SC** ([*ShellCheck*](https://www.shellcheck.net/)): análisis de los comandos dentro de instrucciones `RUN`, con las mismas reglas que aplicaría *ShellCheck* a un *script* de *shell*.

### Qué detecta

| Categoría | Ejemplo de problema | Regla |
|---|---|---|
| Imagen base sin versión fija | `FROM python:latest` | `DL3007` |
| Instalación sin versión de paquete | `apt-get install curl` | `DL3008` |
| `apt-get` sin `--no-install-recommends` | instala dependencias innecesarias | `DL3015` |
| `ADD` en lugar de `COPY` | `ADD` puede desempacar tarballs silenciosamente | `DL3020` |
| Uso de `sudo` en contenedor | eleva privilegios de forma innecesaria | `DL3004` |
| `RUN` encadenado sin limpiar caché | aumenta el tamaño de la imagen | `DL3009` |
| `pip install` sin `--no-cache-dir` | retiene *cache* de paquetes en la imagen | `DL3042` |

### Por qué pertenece al *pipeline* de *build*

*Hadolint* actúa **antes de construir la imagen**: es más barato corregir el *Dockerfile* que escanear una imagen mal construida. El patrón del curso es:

```
lint Dockerfile  →  build imagen  →  scan imagen  →  push al registro
```

Si *Hadolint* falla, el *pipeline* se detiene y no se gasta tiempo ni recursos en construir una imagen que ya tiene problemas conocidos.

### Cómo se invoca

*Hadolint* se distribuye como imagen [*Docker*](https://docs.docker.com/) oficial. No requiere instalación:

```bash
docker run --rm -i hadolint/hadolint:v2.12.0 < Dockerfile
```

Para ignorar una regla específica (p. ej. `DL3008` cuando los paquetes del sistema operativo se gestionan por política externa):

```bash
docker run --rm -i hadolint/hadolint:v2.12.0 hadolint --ignore DL3008 - < Dockerfile
```


## Ejemplo de *workflow* base endurecido.```yamlname: container-buildon:  pull_request:  push:    branches: ["main"]permissions:  contents: read  packages: write  id-token: writejobs:  build-image:    runs-on: ubuntu-latest    steps:      - name: Checkout        uses: actions/checkout@v6      - name: Set up Docker Buildx        uses: docker/setup-buildx-action@v3      - name: Lint Dockerfile        run: docker run --rm -i hadolint/hadolint:v2.12.0 < Dockerfile      - name: Build image        run: docker build -t ghcr.io/org/app:${{ github.sha }} .      - name: Scan image        run: docker run --rm -v /var/run/docker.sock:/var/run/docker.sock goodwithtech/dockle:v0.4.14 ghcr.io/org/app:${{ github.sha }}      - name: Push image (only on main)        if: github.ref == 'refs/heads/main'        run: echo "push step here"```

El paso `Scan image` usa [*Dockle*](https://github.com/goodwithtech/dockle), un inspector de buenas prácticas de seguridad en imágenes *Docker*. Complementa *Trivy* verificando la configuración del contenedor en lugar de *CVEs* en dependencias: detecta imágenes que corren como `root`, exponen puertos innecesarios o incluyen herramientas de depuración en producción.

In [ ]:
import pathlib

# Lee el workflow de build real del submódulo de la aplicación
wf = pathlib.Path("apps/api-github-actions-demo/.github/workflows/envia-a-docker.yaml")
print(wf.read_text())


## Cuándo mover un *job* a self-hosted.Checklist de decisión:- ¿Necesita acceso a red privada no expuesta a internet?- ¿Requiere herramientas/licencias no disponibles en hosted?- ¿Hay requisitos regulatorios de residencia o aislamiento?- ¿Existe capacidad real para operar y endurecer esos *runners*?Si la respuesta es no para estos puntos, mantener hosted suele ser más seguro y simple.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>